# Import thư viện cần thiết

In [1]:
import os
import json
import random
import pandas as pd
from datasets import load_dataset, Dataset
from functools import reduce

/Users/macbook/Desktop/DS304_perfume_investigation/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Set biến global

In [2]:
# Biến Global
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")  # set token qua biến môi trường
HF_REPO_TEXT_ID = "UngLong/openm3chest-labels-v2"
HF_REPO_OUTPUT  = "rimine/cardiology_ready_finetune" # your repo


# Hàm load các dataset

In [7]:
def _build_dataframe(subset_list, repo_input):
    """
    Gop nhieu task (inner join tren pids+keys) -> prompt co constraint + response JSON.
    Tra ve DataFrame (keys, pids, prompt, response). KHONG push.
    """
    dfs = []
    for i, subset in enumerate(subset_list):
        print(f"Loading {subset}...")
        df = pd.DataFrame(load_dataset(repo_input, split="train", name=subset))
        df = df.drop_duplicates(subset=["pids", "keys"], keep="first")
        df = df.rename(columns={"labels": f"label_{subset}",
                                "questions": f"ques_{subset}",
                                "answer_dict": f"ans_{subset}"})
        cols = ["keys", "pids", f"label_{subset}", f"ques_{subset}", f"ans_{subset}"]
        if i == 0:
            cols.insert(2, "clinical_data")
        dfs.append(df[cols])

    merged = reduce(lambda l, r: pd.merge(l, r, on=["pids", "keys"], how="inner"), dfs)
    print(f"{len(subset_list)} tasks -> {len(merged)} scans sau inner join")

    prompts, responses = [], []
    for _, row in merged.iterrows():
        clinical = clinical_to_text(json.loads(row["clinical_data"]))

        q_lines, ans_obj = [], {}
        for subset in subset_list:
            q = random.choice(row[f"ques_{subset}"])
            ans_dict = json.loads(row[f"ans_{subset}"])
            allowed = " | ".join(ans_dict.values())
            q_lines.append(f'- "{subset}": {q} (allowed: {allowed})')
            ans_obj[subset] = ans_dict.get(str(row[f"label_{subset}"]), "Unknown")

        keys_str = ", ".join(f'"{s}"' for s in subset_list)
        prompt = f"""You are a helpful medical assistant. Analyze the provided chest CT scan slices together with the patient's clinical record.

[PATIENT CLINICAL RECORD]
{clinical}

[QUESTIONS]
For each item below, choose exactly one of its allowed values:
{chr(10).join(q_lines)}

[OUTPUT FORMAT]
Respond ONLY with a single JSON object (no extra text) using exactly these keys: {{{keys_str}}}."""

        prompts.append(prompt)
        responses.append(json.dumps(ans_obj, ensure_ascii=False))

    merged["prompt"] = prompts
    merged["response"] = responses
    return merged[["keys", "pids", "prompt", "response"]]


def build_and_upload_dataset(subset_list, repo_input, repo_output):
    """
    Build (qua _build_dataframe) roi push len HF.
    Dung cho agent don gian (1-2 task), KHONG chia pool.
    VD: Cardiology = ["CVD_diagnosis", "CVD_mortality"], Oncology = ["lung_cancer_risk"].
    """
    df = _build_dataframe(subset_list, repo_input)
    final = Dataset.from_pandas(df)
    print(f"Tong so ban ghi: {len(final)}")
    final.push_to_hub(repo_id=repo_output, private=False)
    print(f"Da push len: {repo_output}")
    return final


# Luồng để xử lý và upload lên HF

## Hàm xử lý clinical text thành đoạn văn

In [8]:
import json

def _valid(v):
    """False neu la missing sentinel: None, "", "None", hoac so <= 0."""
    if v is None:
        return False
    if isinstance(v, str):
        return v not in ("", "None")
    if isinstance(v, (int, float)):
        return v > 0
    return True


def clinical_to_text(clinical_data) -> str:
    """
    Convert structured clinical JSON -> plain English paragraph.
    Bo qua missing sentinel: -1.0 (so), "None"/"" (chuoi).
    """
    if isinstance(clinical_data, str):
        clinical_data = json.loads(clinical_data)

    demo    = clinical_data.get("demo", {}) or {}
    smoking = clinical_data.get("smoking", {}) or {}
    disease = clinical_data.get("disease_his", {}) or {}
    cancer  = clinical_data.get("cancer_his", {}) or {}
    fam     = clinical_data.get("fam_lc", {}) or {}

    parts = []

    # Demographics
    if demo:
        tokens = []
        age    = demo.get("age")
        gender = demo.get("gender", "") if _valid(demo.get("gender")) else ""
        race   = demo.get("race")
        ethnic = demo.get("ethnic")
        educat = demo.get("educat")
        height = demo.get("height")
        weight = demo.get("weight")

        if _valid(age):
            tokens.append(f"{int(age)}-year-old {gender}".strip())
        if _valid(race):
            tokens.append(race)
        if _valid(ethnic) and ethnic != "Neither Hispanic nor Latino":
            tokens.append(ethnic)
        if _valid(height) and _valid(weight):
            bmi = round(weight * 0.453592 / ((height * 0.0254) ** 2), 1)
            tokens.append(f"height {int(height)}in, weight {int(weight)}lbs (BMI {bmi})")
        if _valid(educat):
            tokens.append(f"education: {educat}")
        if tokens:
            parts.append("Patient: " + ", ".join(tokens) + ".")

    # Smoking
    status = smoking.get("cigsmok", "")
    if _valid(status) and status != "Never":
        smk      = f"Smoking: {status}"
        pkyr     = smoking.get("pkyr")
        start    = smoking.get("smokeage")
        quit_age = smoking.get("age_quit")
        smokeday = smoking.get("smokeday")
        smokeyr  = smoking.get("smokeyr")
        live     = smoking.get("smokelive", "")
        work     = smoking.get("smokework", "")
        cigar    = smoking.get("cigar", "")
        pipe     = smoking.get("pipe", "")

        if _valid(pkyr):
            smk += f", {int(pkyr)} pack-years"
        if _valid(start):
            smk += f", started age {int(start)}"
        if _valid(smokeday):
            smk += f", {int(smokeday)} cigarettes/day"
        if _valid(smokeyr):
            smk += f", {int(smokeyr)} years"
        if status == "Former" and _valid(quit_age):
            smk += f", quit age {int(quit_age)}"
        if live == "Yes":
            smk += ", lives with smoker"
        if work == "Yes":
            smk += ", works with smoker"
        if cigar == "Yes":
            smk += ", cigar smoker"
        if pipe == "Yes":
            smk += ", pipe smoker"
        parts.append(smk + ".")

    # Disease history
    if disease:
        items = []
        for name, age in disease.items():
            if _valid(age):
                items.append(f"{name} (onset age {int(age)})")
            else:
                items.append(f"{name} (onset age unknown)")
        parts.append(f"Medical history: {', '.join(items)}.")

    # Cancer history
    if cancer:
        items = []
        for name, age in cancer.items():
            if _valid(age):
                items.append(f"{name} (age {int(age)})")
            else:
                items.append(name)
        parts.append(f"Cancer history: {', '.join(items)}.")

    # Family lung cancer history
    if fam:
        relatives = ", ".join(str(k) for k in fam.keys())
        parts.append(f"Family history of lung cancer: {relatives}.")

    return " ".join(parts) if parts else "No clinical data available."


## Demo prompt, response

## (Tuy chon) Xem thu sample truoc khi push
Chay cell nay de kiem tra prompt/response — KHONG push len Hub.

In [9]:
# Preview — build & xem sample, KHONG push
my_subsets = ["CVD_diagnosis", "CVD_mortality"]

df_preview = _build_dataframe(my_subsets, HF_REPO_TEXT_ID)
print(f"\nTong: {len(df_preview)} samples\n")

for i in range(2):  # doi so de xem nhieu hon
    row = df_preview.iloc[i]
    print(f"{'='*70}\n[Sample {i}]  pid={row['pids']}  key={row['keys'][:30]}...\n{'='*70}")
    print("-- PROMPT --")
    print(row["prompt"])
    print("\n-- RESPONSE --")
    print(row["response"], "\n")


Loading CVD_diagnosis...


Generating test split: 100%|██████████| 3759/3759 [00:00<00:00, 201351.02 examples/s]


Loading CVD_mortality...


Generating test split: 100%|██████████| 3759/3759 [00:00<00:00, 146938.82 examples/s]


2 tasks -> 1500 scans sau inner join

Tong: 1500 samples

[Sample 0]  pid=100088  key=1.2.840.113654.2.55.3658236997...
-- PROMPT --
You are a helpful medical assistant. Analyze the provided chest CT scan slices together with the patient's clinical record.

[PATIENT CLINICAL RECORD]
Patient: 70-year-old Female, White, Hispanic or Latino, height 62in, weight 175lbs (BMI 32.0), education: Associate degree/ some college. Smoking: Former, 61 pack-years, started age 16, 30 cigarettes/day, 41 years, quit age 57, lives with smoker, works with smoker.

[QUESTIONS]
For each item below, choose exactly one of its allowed values:
- "CVD_diagnosis": Can you identify any notable abnormalities in the cardiovascular system? (allowed: No | Yes)
- "CVD_mortality": Could you estimate the mortality risk associated with cardiovascular disease? (allowed: Low risk | High risk)

[OUTPUT FORMAT]
Respond ONLY with a single JSON object (no extra text) using exactly these keys: {"CVD_diagnosis", "CVD_mortality"}.

## Tiến hành xử lý

In [ ]:
# Danh sach task can gop (agent don gian, KHONG pool).
# VD Cardiology: ["CVD_diagnosis", "CVD_mortality"]  |  Oncology: ["lung_cancer_risk"]
my_subsets = ["CVD_diagnosis", "CVD_mortality"]

ds = build_and_upload_dataset(my_subsets, HF_REPO_TEXT_ID, HF_REPO_OUTPUT)

# Xem thu sample dau tien
print("\n--- SAMPLE PROMPT ---")
print(ds[0]["prompt"])
print("\n--- SAMPLE RESPONSE ---")
print(ds[0]["response"])
